一、 核心教学内容
这节课的重点在于“如何通过代码进行编排（Orchestration）”，即不再依赖单一的智能体，而是通过 Python 代码将多个专业化的 Agent 串联起来，形成一个完整的业务闭环：

多 Agent 协作系统 (Multi-Agent Orchestration)：
构建了四个功能分立的 Agent：

Planner Agent (规划者)：将大问题拆解为多个具体的搜索任务。

Search Agent (搜索者)：执行具体的网络搜索并提取信息。

Writer Agent (撰写者)：将搜索结果整理并编写为长篇 Markdown 报告。

Emailer Agent (邮件发送者)：将报告转化为 HTML 邮件并通过工具发送。

结构化输出 (Structured Outputs)：
利用 Pydantic 定义数据类（如 WebSearchPlan, ReportData），强制模型输出符合特定格式的 JSON，确保 Agent 之间的数据传输是可靠且无歧义的。

编排函数设计：
学习了如何通过 Python 的 asyncio 异步执行机制来处理并行的搜索任务（run_searches），并按逻辑先后顺序调用不同的 Runner.run() 来衔接各个 Agent 的工作。

云端工具的应用：
介绍了 WebSearchTool 等托管工具的使用，并特别提醒了 API 调用成本与企业级开发的依赖性权衡。

In [2]:
import sys
from pathlib import Path
import os

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

# 检查 api key

api_configs = {
    "deepseek_api_key": "DEEPSEEK_API_KEY",
    "bigmodel_api_key": "BIGMODEL_API_KEY",
    "moonshot_api_key": "MOONSHOT_API_KEY",
    "gemini_api_key": "GEMINI_API_KEY",
    "qianfan_api_key": "QIANFAN_API_KEY",
    "dashscope_api_key": "DASHSCOPE_API_KEY",
    "openai_api_key": "OPENAI_API_KEY",
}

for name, env in api_configs.items():
    key = os.getenv(env)
    if key:
        logger.info(f"{name}: {key[:10]}")
    else:
        raise RuntimeError(f"{env} is not set")


DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
QIANFAN_BASE_URL = "https://qianfan.baidubce.com/v2"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MOONSHOT_BASE_URL = "https://api.moonshot.cn/v1"
BIGMODEL_BASE_URL = "https://open.bigmodel.cn/api/paas/v4"

2026-07-01 13:26:03,932 [INFO] __main__: deepseek_api_key: sk-5837625
2026-07-01 13:26:03,933 [INFO] __main__: bigmodel_api_key: 50b41d629f
2026-07-01 13:26:03,934 [INFO] __main__: moonshot_api_key: sk-0e1imqn
2026-07-01 13:26:03,934 [INFO] __main__: gemini_api_key: AIzaSyDpOi
2026-07-01 13:26:03,934 [INFO] __main__: qianfan_api_key: bce-v3/ALT
2026-07-01 13:26:03,935 [INFO] __main__: dashscope_api_key: sk-3789376
2026-07-01 13:26:03,936 [INFO] __main__: openai_api_key: sk-proj-0I


In [3]:
MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

$\color{yellow}\text{Search Agent}$

In [5]:
from agents import Agent, Runner, WebSearchTool
from agents.model_settings import ModelSettings

# tool_choice
# auto 适合聊天, 由模型自主决定是否调用工具
# required, 作为一个专用工具模型, 不允许空转, 必须调用模型
model_settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

search_instructions = """你是一名研究助理，给定一个搜索词，请在网络上搜索该词，并生成一份简洁的搜索结果摘要。
摘要必须控制在 2-3 段，且总字数少于 300 字。
请抓住要点并保持精炼，回复时仅包含摘要内容，无需其他废话。
"""

search_agent = Agent(
    name="search agent",
    instructions=search_instructions,
    model=MODEL_NAME,
    tools=tools,
    model_settings=model_settings,
)

In [ ]:
search_task = "搜索2026年最受欢迎的 AI Agent 框架"
result = await Runner.run(starting_agent=search_agent, input=search_task)

logger.info(f"\n\n{result.final_output}")

2026-06-30 22:50:12,090 [INFO] __main__: 2026年最受欢迎的 AI Agent 框架，主流讨论集中在 **LangGraph、CrewAI、AutoGen（及其后续的 Microsoft Agent Framework）**。多篇 2026 年的行业综述都把这三者列为“最常被提及/最常用”的核心框架，其中 LangGraph 常被视为复杂、有状态工作流的首选，CrewAI 偏向上手快的多智能体编排，AutoGen 则以对话式多代理协作为代表。([forasoft.com](https://www.forasoft.com/learn/ai-for-video-engineering/articles-ai/langgraph-vs-crewai-vs-autogen-agent-frameworks?utm_source=openai))

如果按“热门 + 生产落地”综合看，LangGraph 往往排名更靠前；而从开发者生态和话题热度看，CrewAI 和 AutoGen 也非常强势。2026 年的趋势还包括：OpenAI Agents SDK、Claude Agent SDK、Semantic Kernel、LlamaIndex Agents、Pydantic AI 等快速升温，但它们更多是在细分场景中流行，而非全面取代前三者。([alicelabs.ai](https://alicelabs.ai/en/insights/best-ai-agent-frameworks-2026?utm_source=openai))


$\color{yellow}\text{Planner Agent}$

In [11]:
from pydantic import BaseModel, Field


class WebSearchItem(BaseModel):
    reason: str = Field(
        description="你进行此次搜索对于完成整体研究任务的重要性的逻辑推理"
    )
    query: str = Field(description="用于网络搜索的实际搜索关键词")


class WebSearchPlan(BaseModel):
    searchs: list[WebSearchItem] = Field(
        description="需要执行的一系列网络搜索的任务清单"
    )


plan_instructions = f"""你是一名研究助理。针对用户的查询，请构思出一组网络搜索任务，以最有效地回答该问题。请输出 {HOW_MANY_SEARCHES} 个用于搜索的关键词。
"""

plan_agent = Agent(name="plan agent", model=MODEL_NAME, output_type=WebSearchPlan)

In [12]:
result = await Runner.run(starting_agent=plan_agent, input=search_task)

logger.info(f"\n\n{result.final_output}")

2026-06-30 23:24:34,880 [INFO] __main__: 

searchs=[WebSearchItem(reason='需要先确认 2026 年的最新市场热度与社区活跃度，避免沿用过时排名。', query='2026 most popular AI agent frameworks community adoption GitHub stars survey'), WebSearchItem(reason='需要对比主流框架在 2026 年的使用情况、生态和企业落地案例。', query='2026 AI agent frameworks comparison LangGraph AutoGen CrewAI Semantic Kernel LlamaIndex popularity'), WebSearchItem(reason='需要了解中文互联网对 2026 年 AI Agent 框架的讨论与推荐。', query='2026 中文 AI Agent 框架 热门 排名 推荐'), WebSearchItem(reason='需要查找权威榜单或技术媒体对 2026 年 AI Agent 框架的评测。', query='2026 best AI agent frameworks review ranking tech media')]


$\color{yellow}\text{Writer Agent}$

In [ ]:
class ReportData(BaseModel):
    short_summary: str = Field(description="2~3句简短的调查结果总结")
    markdown_report: str = Field(description="最终研究报告")
    follow_up_questions: str = Field(description="建议进一步调研的主题")


write_instructions = """你是一名资深研究员，负责针对研究查询撰写一份连贯的报告。
你将获得原始查询信息及相关的研究资料。
请基于这些研究资料和原始查询，撰写一份详尽的综合报告。
最终输出格式应为 Markdown，内容要详实且深入。目标是撰写 5-10 页的内容，字数不少于 1000 字。
"""

write_agent = Agent(name="write agent", model=MODEL_NAME, output_type=ReportData)

$\color{yellow}\text{Email Agent}$

In [6]:
def send_email(subject: str, text_body: str, html_body: str):
    if USE_EMAIL:
        print(f"\n\nsend email: {html_body}")
    else:
        print(f"\n\npush email: {text_body}")


from agents import function_tool


@function_tool
def send_eamil_tool(subject: str, text_body: str, html_body: str):
    """
    使用给定的主题和正文给所有的潜在销售机会发送一封邮件

    Args:
        subject: "邮件的主题"
        text_body: "文本格式的邮件正文"
        html_body: "html 格式的邮件正文"
    """
    send_email(subject, text_body, html_body)
    return "succeed to send email"


send_email_instructions = """你会得到一份详细的报告。使用你的工具发送一封邮件，将报告转换为具有适当主题行的干净的，美观的HTML电子邮件。"""

send_eamil_agent = Agent(
    name="send email agent",
    instructions=send_email_instructions,
    model=MODEL_NAME,
    tools=[send_eamil_tool],
)

send_eamil_tool.params_json_schema

{'properties': {'subject': {'description': '"邮件的主题"',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': '"文本格式的邮件正文"',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': '"html 格式的邮件正文"',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_eamil_tool_args',
 'type': 'object',
 'additionalProperties': False}

$\color{yellow}\text{Orchestrate by Code}$

In [ ]:
async def search(item: WebSearchItem):
    task = f"Reason for searching: \n\n{item.reason}\n\nSearch term: {item.query}"
    result = await Runner.run(starting_agent=search_agent, input=task)

    return result.final_output